# 🛠️ Module 09: Tools & Agents

---

## What Are Agents?

**Agents** are LLMs that can **take actions** — they decide which tools to use, call them, observe results, and repeat until the task is complete. This is called the **ReAct pattern** (Reasoning + Acting).

```
User: "What's the weather in Tokyo and what time is it there?"
  ↓
Agent: "I need to use weather_tool and timezone_tool"
  ↓
[Call weather_tool(Tokyo)] → "18°C, sunny"
[Call timezone_tool(Tokyo)] → "JST, UTC+9, 14:30"
  ↓
Agent: "It's 14:30 JST in Tokyo and the weather is 18°C and sunny."
```

---

## Agent vs Chain

| | Chain | Agent |
|-|-------|-------|
| **Control flow** | Fixed, predefined | Dynamic, LLM decides |
| **Tools** | None (or fixed set) | Chooses from available tools |
| **Loops** | No | Yes (until done) |
| **Use case** | Predictable workflows | Open-ended tasks |
| **Risk** | Low | Higher (unpredictable) |

---

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq

# Agents work better with more capable models
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("Setup complete ✅")

Setup complete ✅


## 1️⃣ Creating Custom Tools

In [2]:
from langchain_core.tools import tool

# ============================================================
# Method 1: @tool decorator — Simplest way
# The docstring becomes the tool's description for the LLM!
# ============================================================

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result.
    Input should be a valid Python math expression like '2 + 3 * 4' or 'sqrt(16)'.
    """
    import math
    try:
        # Safe evaluation of math expressions
        allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_word_count(text: str) -> str:
    """
    Count the number of words in the provided text.
    Returns word count and character count.
    """
    words = text.split()
    return f"Words: {len(words)}, Characters: {len(text)}"

@tool
def get_current_time(timezone: str = "UTC") -> str:
    """
    Get the current date and time.
    Optionally specify timezone (default: UTC).
    Supported: UTC, EST, PST, GMT, IST, JST
    """
    from datetime import datetime
    offsets = {"UTC": 0, "EST": -5, "PST": -8, "GMT": 0, "IST": 5.5, "JST": 9}
    offset = offsets.get(timezone.upper(), 0)
    from datetime import timezone as tz, timedelta
    time = datetime.now(tz.utc) + timedelta(hours=offset)
    return f"Current time in {timezone}: {time.strftime('%Y-%m-%d %H:%M:%S')}"

# Inspect the tools
tools = [calculator, get_word_count, get_current_time]

for tool_fn in tools:
    print(f"Tool: {tool_fn.name}")
    print(f"Description: {tool_fn.description[:80]}...")
    print(f"Schema: {tool_fn.args_schema.model_json_schema()}")
    print()

Tool: calculator
Description: Evaluate a mathematical expression and return the result.
Input should be a vali...
Schema: {'description': "Evaluate a mathematical expression and return the result.\nInput should be a valid Python math expression like '2 + 3 * 4' or 'sqrt(16)'.", 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}

Tool: get_word_count
Description: Count the number of words in the provided text.
Returns word count and character...
Schema: {'description': 'Count the number of words in the provided text.\nReturns word count and character count.', 'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'get_word_count', 'type': 'object'}

Tool: get_current_time
Description: Get the current date and time.
Optionally specify timezone (default: UTC).
Suppo...
Schema: {'description': 'Get the current date and time.\nOptionally specify timezone (default: 

In [3]:
# ============================================================
# Method 2: StructuredTool with Pydantic input schema
# ============================================================
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(description="City name to get weather for")
    unit: str = Field(default="celsius", description="Temperature unit: celsius or fahrenheit")

def get_weather_impl(city: str, unit: str = "celsius") -> str:
    """Simulated weather data (replace with real API)"""
    weather_data = {
        "New York": {"temp": 22, "condition": "sunny", "humidity": 65},
        "London": {"temp": 15, "condition": "cloudy", "humidity": 80},
        "Tokyo": {"temp": 28, "condition": "partly cloudy", "humidity": 70},
        "Paris": {"temp": 18, "condition": "rainy", "humidity": 85},
    }
    
    data = weather_data.get(city, {"temp": 20, "condition": "unknown", "humidity": 60})
    temp = data["temp"]
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    
    unit_symbol = "°F" if unit == "fahrenheit" else "°C"
    return f"{city}: {temp}{unit_symbol}, {data['condition']}, humidity {data['humidity']}%"

weather_tool = StructuredTool.from_function(
    func=get_weather_impl,
    name="get_weather",
    description="Get current weather for a city. Returns temperature, conditions, and humidity.",
    args_schema=WeatherInput
)

# Test the tool directly
print(weather_tool.invoke({"city": "Tokyo", "unit": "celsius"}))
print(weather_tool.invoke({"city": "New York", "unit": "fahrenheit"}))

Tokyo: 28°C, partly cloudy, humidity 70%
New York: 71.6°F, sunny, humidity 65%


## 2️⃣ Built-in Tools

In [4]:
# ============================================================
# Built-in tools from LangChain Community
# ============================================================

# DuckDuckGo Search (free, no API key needed!)
# %pip install -q duckduckgo-search
# from langchain_community.tools import DuckDuckGoSearchRun
# search_tool = DuckDuckGoSearchRun()
# result = search_tool.run("LangChain latest version 2024")
# print(result)

# Wikipedia
# %pip install -q wikipedia
# from langchain_community.tools import WikipediaQueryRun
# from langchain_community.utilities import WikipediaAPIWrapper
# wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Python REPL (execute code!)
# from langchain_experimental.tools import PythonREPLTool
# python_tool = PythonREPLTool()

print("Available built-in tools:")
print("  - DuckDuckGoSearchRun (web search)")
print("  - WikipediaQueryRun")
print("  - PythonREPLTool (run Python code)")
print("  - ShellTool (run shell commands)")
print("  - ArXivQueryRun (search papers)")
print("  - OpenWeatherMapQueryRun")
print("  - YahooFinanceNewsTool")
print("  - FileManagementToolkit")

Available built-in tools:
  - DuckDuckGoSearchRun (web search)
  - WikipediaQueryRun
  - PythonREPLTool (run Python code)
  - ShellTool (run shell commands)
  - ArXivQueryRun (search papers)
  - OpenWeatherMapQueryRun
  - YahooFinanceNewsTool
  - FileManagementToolkit


In [11]:
import langchain

print(langchain.__version__)

from langchain.agents import create_react_agent

1.2.15


ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (c:\Users\sujat\projects\AI\.venv\Lib\site-packages\langchain\agents\__init__.py)

## 3️⃣ Building an Agent with create_react_agent

In [9]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

# ============================================================
# Create a ReAct agent
# ReAct = Reasoning + Acting
# ============================================================

# Get the standard ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")
print("ReAct prompt template:")
print(prompt.template[:500])

ImportError: cannot import name 'hub' from 'langchain' (c:\Users\sujat\projects\AI\.venv\Lib\site-packages\langchain\__init__.py)

In [ ]:
# All tools our agent can use
all_tools = [calculator, get_word_count, get_current_time, weather_tool]

# Create the agent
agent = create_react_agent(llm, all_tools, prompt)

# Wrap in AgentExecutor for execution
agent_executor = AgentExecutor(
    agent=agent,
    tools=all_tools,
    verbose=True,          # Show reasoning steps
    max_iterations=5,      # Limit loop iterations
    handle_parsing_errors=True  # Handle LLM format errors
)

print("Agent ready! ✅")

In [ ]:
# ============================================================
# Run the agent!
# ============================================================
result = agent_executor.invoke({
    "input": "What is 15 * 7 + sqrt(49)? Also, what time is it in Tokyo right now?"
})

print("\n" + "="*60)
print("FINAL ANSWER:")
print(result["output"])

In [ ]:
# Multi-step reasoning
result = agent_executor.invoke({
    "input": "What's the weather in London and Paris? Which city is warmer?"
})

print("\nFINAL ANSWER:")
print(result["output"])

## 4️⃣ Tool Calling Agent (Modern Approach)

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ============================================================
# Tool Calling Agent uses OpenAI function calling
# More reliable than ReAct for structured tool use!
# ============================================================

prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful assistant with access to various tools.
    Use the available tools to answer user questions accurately.
    Always show your work when doing calculations.
    """),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")  # Required for tool calls
])

# Create the tool-calling agent
agent = create_tool_calling_agent(llm, all_tools, prompt)

executor = AgentExecutor(
    agent=agent,
    tools=all_tools,
    verbose=True,
    max_iterations=10
)

# Complex multi-tool task
result = executor.invoke({
    "input": """
    Help me with these tasks:
    1. Calculate the area of a circle with radius 7 (pi * r^2)
    2. Count the words in this sentence: 'The quick brown fox jumps over the lazy dog'
    3. What's the current UTC time?
    """
})

print("\n" + "="*60)
print("FINAL ANSWER:")
print(result["output"])

## 5️⃣ Agent with Memory

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# ============================================================
# Add memory to the agent
# ============================================================

store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    executor,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

config = {"configurable": {"session_id": "agent_session"}}

# Turn 1: Ask something
r1 = agent_with_memory.invoke({"input": "What's 25 * 4?"}, config=config)
print(f"Turn 1: {r1['output']}\n")

# Turn 2: Follow-up using memory
r2 = agent_with_memory.invoke({"input": "Now square that result."}, config=config)
print(f"Turn 2: {r2['output']}")

## 6️⃣ Building Custom Agent Logic

In [ ]:
# ============================================================
# Router Agent — Classify and route to specialized sub-agents
# ============================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Specialized agents
math_agent = AgentExecutor(
    agent=create_tool_calling_agent(llm, [calculator], ChatPromptTemplate.from_messages([
        ("system", "You are a math expert. Only use the calculator tool."),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad")
    ])),
    tools=[calculator]
)

weather_agent = AgentExecutor(
    agent=create_tool_calling_agent(llm, [weather_tool, get_current_time], ChatPromptTemplate.from_messages([
        ("system", "You are a weather and time assistant."),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad")
    ])),
    tools=[weather_tool, get_current_time]
)

# Router chain
router_prompt = ChatPromptTemplate.from_template(
    """Classify this question into exactly one category:
    - MATH: mathematical calculations
    - WEATHER: weather or time questions  
    - GENERAL: everything else
    
    Question: {question}
    Category (just the word):"""
)

router = router_prompt | llm | StrOutputParser()

def route_to_agent(question: str) -> str:
    """Route question to the right specialist agent"""
    category = router.invoke({"question": question}).strip().upper()
    print(f"  → Routing to: {category} agent")
    
    if category == "MATH":
        result = math_agent.invoke({"input": question})
    elif category == "WEATHER":
        result = weather_agent.invoke({"input": question})
    else:
        result = llm.invoke(question).content
        return result
    
    return result["output"]

# Test the router
questions = [
    "What is 144 / 12 + 7^2?",
    "What's the weather like in London?",
    "What is the history of the internet?"
]

for q in questions:
    print(f"\n❓ {q}")
    answer = route_to_agent(q)
    print(f"💬 {answer}")

## ✅ Module 09 Summary

You've learned:
- ✅ What agents are and when to use them (vs chains)
- ✅ Creating tools with `@tool` decorator
- ✅ `StructuredTool` with Pydantic schemas
- ✅ Built-in LangChain tools
- ✅ ReAct agent pattern
- ✅ Tool-calling agent (modern, recommended)
- ✅ Agent with memory
- ✅ Router agent pattern

### 🚀 Next: [Module 10 — LangGraph: Multi-Agent Workflows](10_LangGraph.ipynb)